<a href="https://colab.research.google.com/github/OlajideFemi/Carbon-Footprint/blob/main/Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Simple sentiment analysis model.

Usage:
    from sentiment import analyze_sentiment
    import pandas as pd

    df = pd.DataFrame({"text": ["I love this!", "This is terrible.", "It's okay."]})
    result = analyze_sentiment(df, text_col="text")
    print(result)

Approach:
  - Lexicon-based scoring via VADER (works well on short/free text, no training needed).
  - Falls back to a tiny built-in lexicon if `vaderSentiment` isn't installed.
  - Returns the original df with added columns: neg, neu, pos, compound, sentiment.
"""

from __future__ import annotations
import re
import pandas as pd


def _get_analyzer():
    try:
        from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
        return SentimentIntensityAnalyzer()
    except ImportError:
        return _FallbackAnalyzer()


class _FallbackAnalyzer:
    """Tiny lexicon fallback if vaderSentiment isn't installed."""
    POS = {"good","great","love","excellent","amazing","awesome","happy","best",
           "fantastic","wonderful","nice","like","enjoy","perfect","superb"}
    NEG = {"bad","terrible","hate","awful","worst","horrible","sad","poor",
           "disappointing","boring","ugly","dislike","broken","angry","annoying"}

    def polarity_scores(self, text: str):
        tokens = re.findall(r"[a-zA-Z']+", str(text).lower())
        if not tokens:
            return {"neg":0.0,"neu":1.0,"pos":0.0,"compound":0.0}
        pos = sum(t in self.POS for t in tokens)
        neg = sum(t in self.NEG for t in tokens)
        total = len(tokens)
        p, n = pos/total, neg/total
        neu = max(0.0, 1.0 - p - n)
        compound = max(-1.0, min(1.0, (pos - neg) / (pos + neg + 1)))
        return {"neg":n,"neu":neu,"pos":p,"compound":compound}


def _label(compound: float) -> str:
    if compound >= 0.05:
        return "positive"
    if compound <= -0.05:
        return "negative"
    return "neutral"


def analyze_sentiment(df: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
    """Score each row's free text; returns df + neg/neu/pos/compound/sentiment."""
    if text_col not in df.columns:
        raise KeyError(f"Column '{text_col}' not found. Available: {list(df.columns)}")

    analyzer = _get_analyzer()
    scores = df[text_col].fillna("").astype(str).apply(analyzer.polarity_scores)
    scores_df = pd.DataFrame(list(scores), index=df.index)
    scores_df["sentiment"] = scores_df["compound"].apply(_label)
    return pd.concat([df.reset_index(drop=True), scores_df.reset_index(drop=True)], axis=1)


if __name__ == "__main__":
    demo = pd.DataFrame({"text": [
        "I absolutely love this product, it's fantastic!",
        "Worst experience ever. Totally disappointing and awful.",
        "It arrived on Tuesday.",
        "Pretty good, but a bit boring in places.",
        "",
    ]})
    print(analyze_sentiment(demo, "text").to_string(index=False))


                                                   text      neg      neu      pos  compound sentiment
        I absolutely love this product, it's fantastic! 0.000000 0.714286 0.285714  0.666667  positive
Worst experience ever. Totally disappointing and awful. 0.428571 0.571429 0.000000 -0.750000  negative
                                 It arrived on Tuesday. 0.000000 1.000000 0.000000  0.000000   neutral
               Pretty good, but a bit boring in places. 0.125000 0.750000 0.125000  0.000000   neutral
                                                        0.000000 1.000000 0.000000  0.000000   neutral
